# 01 — DAG Anatomy & Scheduling

You've used MWAA in prod (A2H, UIL, LML). This covers the decisions interviewers actually probe.

## Setup — run this first

In [ ]:
# Install Airflow locally (run once)
# !pip install "apache-airflow==2.9.1" --constraint \
#   "https://raw.githubusercontent.com/apache/airflow/constraints-2.9.1/constraints-3.12.txt"

## 1. The skeleton every DAG has

In [ ]:
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

default_args = {
    "owner": "kanishk",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,   # 5m → 10m → 20m
    "email_on_failure": True,
    "email": ["kanishkparganiha@gmail.com"],
}

with DAG(
    dag_id="asin_daily_pipeline",
    default_args=default_args,
    schedule="0 6 * * *",        # 6am UTC daily
    start_date=datetime(2024, 1, 1),
    catchup=False,               # ALWAYS False unless backfill is needed
    max_active_runs=1,           # prevents overlapping runs
    tags=["etl", "asin"],
) as dag:
    pass

print("DAG object created:", dag.dag_id)

## 2. Key interview questions on default_args

**Q: What if `retries=3` in default_args and `retries=1` on a specific task?**

> Task-level config wins. `default_args` are just defaults — any task-level setting overrides them.

---

**Q: `catchup=True`, `start_date` is 30 days ago — what happens on first activation?**

> Airflow immediately creates 30 backfill DAG runs. Usually catastrophic in prod. Always `catchup=False` unless you explicitly need backfill.

---

**Q: `max_active_runs=1` vs `depends_on_past=True` — difference?**

> `max_active_runs=1` → only 1 DAG run at a time globally. `depends_on_past=True` → a task won't run unless the same task in the *previous run* succeeded. Use `max_active_runs=1` for stateful pipelines (your A2H pattern — two runs overlapping would corrupt S3 delta). Use `depends_on_past` for incremental logic that truly depends on prior output.

## 3. Task dependencies

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def extract(ds, **kwargs):   print(f"extract for {ds}")
def validate(**kwargs):      print("validate")
def transform(**kwargs):     print("transform")
def load(**kwargs):          print("load")

with DAG("dependency_demo", start_date=datetime(2024,1,1), catchup=False) as dag:
    t1 = PythonOperator(task_id="extract",   python_callable=extract)
    t2 = PythonOperator(task_id="validate",  python_callable=validate)
    t3 = PythonOperator(task_id="transform", python_callable=transform)
    t4 = PythonOperator(task_id="load",      python_callable=load)

    # Linear chain
    t1 >> t2 >> t3 >> t4

    # Same as:
    # t1.set_downstream(t2)
    # t4.set_upstream(t3)

    # Fan-out then fan-in
    # t1 >> [t2, t3] >> t4

print("Tasks:", [t.task_id for t in dag.tasks])
print("t1 downstream:", [t.task_id for t in t1.downstream_list])

## 4. Cron schedule reference

```
┌─── minute  (0-59)
│ ┌─── hour  (0-23)
│ │ ┌─── day of month  (1-31)
│ │ │ ┌─── month  (1-12)
│ │ │ │ ┌─── day of week  (0-6, Sun=0)
│ │ │ │ │
0 6 * * *   → 6am every day
0 */4 * * * → every 4 hours
0 8 * * 1   → 8am every Monday
0 0 1 * *   → midnight on 1st of each month
```

Presets: `@hourly`, `@daily`, `@weekly`, `@monthly`, `None` (manual only)

## 5. Jinja templates — the ones you actually use

In [ ]:
# These are string templates resolved at runtime — only work in operator string args
# NOT inside Python functions (use ds kwarg instead)

template_examples = {
    "{{ ds }}": "2024-01-15",             # execution date
    "{{ ds_nodash }}": "20240115",         # no dashes
    "{{ ts }}": "2024-01-15T06:00:00+00:00",
    "{{ prev_ds }}": "2024-01-14",
    "{{ next_ds }}": "2024-01-16",
    "{{ dag.dag_id }}": "asin_daily_pipeline",
    "{{ run_id }}": "scheduled__2024-01-15T06:00:00+00:00",
}

for template, example in template_examples.items():
    print(f"{template:35s} → {example}")

In [ ]:
# In practice — your A2H / UIL MWAA pattern:
from airflow.providers.amazon.aws.operators.glue import GlueJobOperator

# GlueJobOperator(
#     task_id="run_asin_glue_job",
#     job_name="asin-classification",
#     script_args={
#         "--date":   "{{ ds }}",
#         "--region": "{{ var.value.region }}",
#         "--env":    "prod",
#     },
#     aws_conn_id="aws_default",
#     wait_for_completion=True,
# )
print("GlueJobOperator template shown above (commented — needs MWAA env)")